# CLIP-ViT + LoRA Continual Learning on Split CIFAR-100

This notebook follows the simplified supervisor-aligned setup.

Main setup:

- CLIP-ViT vision encoder: `openai/clip-vit-base-patch16`
- Split CIFAR-100 continual learning
- 5 steps, 20 classes per step
- Debug first with 1 epoch
- Keep the comparison simple and explainable

Important:

- This is **CLIP-ViT**, not standard supervised ImageNet ViT.
- The CLIP text encoder is not used.
- Only the CLIP vision encoder is used.
- A new CIFAR-100 classifier is trained on top of the CLIP vision representation.
- For CLIP-ViT, LoRA target modules are `q_proj` and `v_proj`.

Main comparison:

1. `seq_ft_no_replay`
2. `simple_avg_no_replay`
3. `simple_avg_replay`
4. `do_merging_simple`
5. `orthogonal_loss`
6. `rank_extension`
7. `joint_upper_bound`

The main method is:

`simple_avg_no_replay`

The replay version is added only beside simple average as a diagnostic:

`simple_avg_replay`

Supervisor-suggested optional methods:

- `orthogonal_loss`: sequential LoRA training where previous LoRA updates are absorbed into the model after each step, and the next LoRA is trained with the supervisor's orthogonality-style loss term.
- `rank_extension`: one LoRA whose rank grows step by step; old rank parts are copied and frozen, and only the new rank block is trained.

Paper-inspired optional method:

- `do_merging_simple`: simplified DO-Merging-inspired LoRA merge. It is not a full paper reproduction yet.


In [ ]:
import os
import gc
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset, concatenate_datasets
from torchvision import transforms

from transformers import (
    CLIPImageProcessor,
    CLIPVisionModel,
    TrainingArguments,
    Trainer,
    set_seed,
)

from transformers.modeling_outputs import ImageClassifierOutput

from peft import LoraConfig, get_peft_model

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

In [ ]:
# ============================================================
# 2) Full comparison configuration
# ============================================================

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEBUG_MODE = False

RUN_NAME = "clip_vit_lora_cifar100_full_comparison_with_orth_rankext"

# CLIP-ViT checkpoint
MODEL_CHECKPOINT = "openai/clip-vit-base-patch16"

NUM_CLASSES = 100
NUM_STEPS = 5
CLASSES_PER_STEP = 20

# ============================================================
# Epoch setup
# ============================================================
# This is the first full comparison setting.
# It is stronger than debug=1, but not too expensive.
FT_EPOCHS = 3
LORA_EPOCHS = 3
JOINT_EPOCHS = 3
ORTH_EPOCHS = 3
RANKEXT_EPOCHS = 3

# Later, for a final heavier run, you can try:
# FT_EPOCHS = 5
# LORA_EPOCHS = 5
# JOINT_EPOCHS = 5
# ORTH_EPOCHS = 5
# RANKEXT_EPOCHS = 5

# ============================================================
# Batch settings
# ============================================================
BATCH_FT = 8
ACCUM_FT = 2

BATCH_LORA = 16
ACCUM_LORA = 1

# ============================================================
# Learning rates
# ============================================================
LR_FT = 3e-5
LR_LORA = 5e-5
LR_JOINT = 5e-5
LR_ORTH = 5e-5
LR_RANKEXT = 5e-5

WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.10
SCHED = "cosine"

USE_FP16 = torch.cuda.is_available()

# ============================================================
# LoRA setup for CLIP-ViT
# ============================================================
# Standard ViT uses: ["query", "value"]
# CLIP-ViT uses:    ["q_proj", "v_proj"]
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]

# ============================================================
# Small replay diagnostic beside simple average
# ============================================================
REPLAY_PER_CLASS = 20

# ============================================================
# Orthogonal-loss setup from supervisor's idea
# ============================================================
# Loss = CE + lambda_orth * mean_i tr(M_(t-1)^i (Delta W_t^i)^T)
# In code we use a normalized trace/dot version for numerical stability.
LAMBDA_ORTH = 0.1

# ============================================================
# Rank-extension setup from Pietro/Leon clarification
# ============================================================
# Step 1 rank=8, step 2 rank=16, step 3 rank=24, ...
RANKEXT_NEW_RANK_PER_STEP = 8

# ============================================================
# Full comparison: run all methods
# ============================================================
METHODS_TO_RUN = {
    "seq_ft_no_replay": True,
    "simple_avg_no_replay": True,
    "simple_avg_replay": True,
    "do_merging_simple": True,
    "orthogonal_loss": True,
    "rank_extension": True,
    "joint_upper_bound": True,
}

BASE_OUTPUT_DIR = os.path.join("outputs", RUN_NAME)
TABLES_DIR = os.path.join(BASE_OUTPUT_DIR, "tables")
PLOTS_DIR = os.path.join(BASE_OUTPUT_DIR, "plots")
MODELS_DIR = os.path.join(BASE_OUTPUT_DIR, "models")

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

all_results = []

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("FP16:", USE_FP16)
print("Checkpoint:", MODEL_CHECKPOINT)
print("Epochs:")
print({
    "FT_EPOCHS": FT_EPOCHS,
    "LORA_EPOCHS": LORA_EPOCHS,
    "JOINT_EPOCHS": JOINT_EPOCHS,
    "ORTH_EPOCHS": ORTH_EPOCHS,
    "RANKEXT_EPOCHS": RANKEXT_EPOCHS,
})
print("Methods:")
print(json.dumps(METHODS_TO_RUN, indent=2))

In [ ]:
# ============================================================
# 3) Load CIFAR-100 and define continual splits
# ============================================================

dataset = load_dataset("cifar100")

# Hugging Face CIFAR-100 usually has columns:
# img, fine_label, coarse_label
LABEL_COL = "fine_label" if "fine_label" in dataset["train"].column_names else "label"
IMAGE_COL = "img" if "img" in dataset["train"].column_names else "image"

class_splits = [
    list(range(0, 20)),
    list(range(20, 40)),
    list(range(40, 60)),
    list(range(60, 80)),
    list(range(80, 100)),
]

first_step_classes = class_splits[0]
later_step_classes = [c for split in class_splits[1:] for c in split]
all_classes = [c for split in class_splits for c in split]

def classes_for_step(step_idx):
    return class_splits[step_idx]

def filter_by_classes(ds, class_ids):
    class_ids = set(class_ids)
    return ds.filter(lambda x: int(x[LABEL_COL]) in class_ids)

print("Dataset columns:", dataset["train"].column_names)
print("Label column:", LABEL_COL)
print("Image column:", IMAGE_COL)
for i, cls in enumerate(class_splits, start=1):
    print(f"Step {i}: {cls[0]}-{cls[-1]}")

In [ ]:
# ============================================================
# 4) CLIP image processor and transforms
# ============================================================

image_processor = CLIPImageProcessor.from_pretrained(MODEL_CHECKPOINT)

if hasattr(image_processor, "crop_size") and image_processor.crop_size is not None:
    H = int(image_processor.crop_size.get("height", 224))
    W = int(image_processor.crop_size.get("width", 224))
else:
    H = W = 224

train_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomCrop((H, W), padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.05,
        contrast=0.05,
        saturation=0.05,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std,
    ),
])

val_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std,
    ),
])

def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")

    if isinstance(x, dict):
        if "array" in x:
            x = x["array"]
        elif "bytes" in x:
            import io
            return Image.open(io.BytesIO(x["bytes"])).convert("RGB")

    if isinstance(x, list):
        x = np.array(x, dtype=np.uint8)

    if isinstance(x, np.ndarray):
        arr = np.squeeze(x).astype(np.uint8)

        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)

        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
            arr = np.transpose(arr, (1, 2, 0))

        if arr.ndim == 3 and arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)

        return Image.fromarray(arr).convert("RGB")

    return x

def preprocess_train(ex):
    ex["pixel_values"] = [train_transform(to_pil(img)) for img in ex[IMAGE_COL]]
    ex["labels"] = [int(y) for y in ex[LABEL_COL]]
    return ex

def preprocess_val(ex):
    ex["pixel_values"] = [val_transform(to_pil(img)) for img in ex[IMAGE_COL]]
    ex["labels"] = [int(y) for y in ex[LABEL_COL]]
    return ex

def collate_fn(examples):
    pixel_values = torch.stack([e["pixel_values"] for e in examples])
    labels = torch.tensor([int(e["labels"]) for e in examples], dtype=torch.long)
    return {
        "pixel_values": pixel_values,
        "labels": labels,
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": float((preds == labels).mean())
    }

print("Image size:", H, W)
print("CLIP mean:", image_processor.image_mean)
print("CLIP std:", image_processor.image_std)

In [ ]:
# ============================================================
# 5) CLIP-ViT vision classifier wrapper
# ============================================================

class CLIPVisionForCIFAR100(nn.Module):
    """
    CLIP-ViT vision encoder + trainable CIFAR-100 classifier.

    This uses:
    openai/clip-vit-base-patch16

    The text encoder is not used.
    Only the CLIP vision backbone is used.
    """

    def __init__(self, checkpoint, num_labels):
        super().__init__()

        self.vision_model = CLIPVisionModel.from_pretrained(checkpoint)

        hidden_size = self.vision_model.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_labels)

        self.config = self.vision_model.config
        self.config.num_labels = num_labels
        self.config.id2label = {i: str(i) for i in range(num_labels)}
        self.config.label2id = {str(i): i for i in range(num_labels)}

    def forward(
        self,
        pixel_values=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=True,
        **kwargs,
    ):
        outputs = self.vision_model(
            pixel_values=pixel_values,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=True,
        )

        pooled_output = outputs.pooler_output
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return ImageClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

def fresh_pretrained_model():
    """
    Fresh CLIP-ViT vision model with a CIFAR-100 classifier.
    """
    return CLIPVisionForCIFAR100(
        checkpoint=MODEL_CHECKPOINT,
        num_labels=NUM_CLASSES,
    )

def add_lora(model):
    """
    Add LoRA to CLIP-ViT attention q_proj and v_proj modules.
    """
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        modules_to_save=["classifier"],
    )

    model = get_peft_model(model, lora_config)
    return model

print("LoRA target modules:", TARGET_MODULES)

In [ ]:
# ============================================================
# 6) Dataset helpers
# ============================================================

def build_replay_dataset(old_classes, replay_per_class):
    if len(old_classes) == 0 or replay_per_class <= 0:
        return None

    parts = []

    for cls in old_classes:
        cls_ds = filter_by_classes(dataset["train"], [cls])
        n = min(replay_per_class, len(cls_ds))
        cls_ds = cls_ds.shuffle(seed=SEED).select(range(n))
        parts.append(cls_ds)

    replay_ds = concatenate_datasets(parts)
    return replay_ds

def make_train_dataset(step_idx, replay_per_class=0):
    current_classes = classes_for_step(step_idx)
    current_ds = filter_by_classes(dataset["train"], current_classes)

    old_classes = []
    for old_step in range(step_idx):
        old_classes.extend(classes_for_step(old_step))

    replay_ds = build_replay_dataset(
        old_classes=old_classes,
        replay_per_class=replay_per_class,
    )

    if replay_ds is None:
        final_ds = current_ds
    else:
        final_ds = concatenate_datasets([current_ds, replay_ds])

    final_ds = final_ds.shuffle(seed=SEED + step_idx)
    final_ds = final_ds.with_transform(preprocess_train)

    print(
        f"Step {step_idx + 1} | "
        f"current={len(current_ds)} | "
        f"replay={0 if replay_ds is None else len(replay_ds)} | "
        f"total={len(final_ds)}"
    )

    return final_ds

def make_eval_dataset(class_ids):
    ds = filter_by_classes(dataset["test"], class_ids)
    ds = ds.with_transform(preprocess_val)
    return ds

def make_joint_train_dataset():
    ds = dataset["train"].shuffle(seed=SEED)
    ds = ds.with_transform(preprocess_train)
    return ds

def make_joint_eval_dataset():
    ds = dataset["test"]
    ds = ds.with_transform(preprocess_val)
    return ds

eval_first = make_eval_dataset(first_step_classes)
eval_later = make_eval_dataset(later_step_classes)
eval_all_seen = make_eval_dataset(all_classes)

print("first_step eval:", len(eval_first))
print("later_steps eval:", len(eval_later))
print("all_seen eval:", len(eval_all_seen))

In [ ]:
# ============================================================
# 7) Trainer helpers
# ============================================================

def get_training_args(
    output_dir,
    epochs,
    lr,
    batch_size,
    accum_steps,
    eval_strategy="epoch",
):
    return TrainingArguments(
        output_dir=output_dir,
        remove_unused_columns=False,
        eval_strategy=eval_strategy,
        save_strategy="no",
        num_train_epochs=epochs,
        learning_rate=lr,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=accum_steps,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        report_to="none",
        max_grad_norm=1.0,
    )

def train_with_trainer(
    model,
    train_ds,
    eval_ds,
    output_dir,
    epochs,
    lr,
    batch_size,
    accum_steps,
    trainer_cls=Trainer,
    **trainer_kwargs,
):
    args = get_training_args(
        output_dir=output_dir,
        epochs=epochs,
        lr=lr,
        batch_size=batch_size,
        accum_steps=accum_steps,
    )

    trainer = trainer_cls(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
        **trainer_kwargs,
    )

    trainer.train()
    eval_out = trainer.evaluate()

    return trainer, eval_out

def evaluate_model(model, method_name):
    args = get_training_args(
        output_dir=os.path.join(MODELS_DIR, f"eval_{method_name}"),
        epochs=1,
        lr=LR_LORA,
        batch_size=BATCH_LORA,
        accum_steps=ACCUM_LORA,
        eval_strategy="no",
    )

    trainer = Trainer(
        model=model,
        args=args,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    eval_first_out = trainer.evaluate(eval_dataset=eval_first)
    eval_later_out = trainer.evaluate(eval_dataset=eval_later)
    eval_all_out = trainer.evaluate(eval_dataset=eval_all_seen)

    rows = [
        {
            "method": method_name,
            "eval_set": "first_step",
            "accuracy": float(eval_first_out["eval_accuracy"]),
            "loss": float(eval_first_out["eval_loss"]),
        },
        {
            "method": method_name,
            "eval_set": "later_steps",
            "accuracy": float(eval_later_out["eval_accuracy"]),
            "loss": float(eval_later_out["eval_loss"]),
        },
        {
            "method": method_name,
            "eval_set": "all_seen",
            "accuracy": float(eval_all_out["eval_accuracy"]),
            "loss": float(eval_all_out["eval_loss"]),
        },
    ]

    all_results.extend(rows)

    print(pd.DataFrame(rows))
    return rows

In [ ]:
# ============================================================
# 8) LoRA extraction and merge helpers
# ============================================================

def normalize_module_name(name):
    prefixes = [
        "base_model.model.",
        "model.",
    ]

    out = name

    for p in prefixes:
        if out.startswith(p):
            out = out[len(p):]

    return out

def extract_lora_state(model):
    """
    Extract:
    - LoRA delta_W per target module
    - classifier weights

    PEFT convention:
    A shape = [r, in_features]
    B shape = [out_features, r]
    delta_W = B @ A * scaling
    """
    state = {
        "deltas": {},
        "classifier_weight": None,
        "classifier_bias": None,
    }

    for name, module in model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        A = module.lora_A["default"].weight.detach().cpu().float()
        B = module.lora_B["default"].weight.detach().cpu().float()

        scaling = (
            module.scaling["default"]
            if isinstance(module.scaling, dict)
            else module.scaling
        )

        delta = (B @ A) * float(scaling)

        plain_name = normalize_module_name(name)
        state["deltas"][plain_name] = delta

    for name, tensor in model.state_dict().items():
        if "classifier.modules_to_save.default.weight" in name:
            state["classifier_weight"] = tensor.detach().cpu().clone()

        if "classifier.modules_to_save.default.bias" in name:
            state["classifier_bias"] = tensor.detach().cpu().clone()

    return state

def get_submodule_by_name(model, module_name):
    module_name = normalize_module_name(module_name)

    current = model

    for part in module_name.split("."):
        if part == "":
            continue
        current = getattr(current, part)

    return current

def simple_average_deltas(step_states):
    keys = sorted(step_states[0]["deltas"].keys())
    merged = {}

    for key in keys:
        vals = []

        for state in step_states:
            if key in state["deltas"]:
                vals.append(state["deltas"][key].float())

        merged[key] = torch.stack(vals, dim=0).mean(dim=0)

    return merged

def do_merge_deltas(step_states):
    """
    Simplified DO-Merging-inspired merge.

    This is not the full paper reproduction.
    It only decouples magnitude and direction before averaging.
    """
    keys = sorted(step_states[0]["deltas"].keys())
    merged = {}

    eps = 1e-8

    for key in keys:
        deltas = []

        for state in step_states:
            if key in state["deltas"]:
                deltas.append(state["deltas"][key].float())

        norms = [torch.linalg.norm(d).clamp_min(eps) for d in deltas]
        directions = [d / n for d, n in zip(deltas, norms)]

        mean_norm = torch.stack(norms).mean()
        mean_direction = torch.stack(directions, dim=0).mean(dim=0)
        mean_direction = mean_direction / torch.linalg.norm(mean_direction).clamp_min(eps)

        merged[key] = mean_norm * mean_direction

    return merged

def apply_deltas_to_base(merged_deltas, step_states):
    """
    Apply merged LoRA deltas to a fresh CLIP-ViT model and stitch classifier rows.
    """
    model = fresh_pretrained_model()

    with torch.no_grad():
        for key, delta in merged_deltas.items():
            try:
                module = get_submodule_by_name(model, key)
            except Exception as e:
                print("Could not find module:", key, "|", e)
                continue

            if not hasattr(module, "weight"):
                print("Module has no weight:", key)
                continue

            module.weight.add_(
                delta.to(
                    device=module.weight.device,
                    dtype=module.weight.dtype,
                )
            )

        for step_idx, state in enumerate(step_states):
            classes = classes_for_step(step_idx)

            if state["classifier_weight"] is None:
                print("Missing classifier for step", step_idx + 1)
                continue

            w = state["classifier_weight"].to(model.classifier.weight.device)
            b = state["classifier_bias"].to(model.classifier.bias.device)

            for c in classes:
                model.classifier.weight[c].copy_(w[c])
                model.classifier.bias[c].copy_(b[c])

    return model

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 9) Baseline: seq_ft_no_replay
# ============================================================

if METHODS_TO_RUN["seq_ft_no_replay"]:
    seq_ft_model = fresh_pretrained_model()

    for step_idx in range(NUM_STEPS):
        train_ds = make_train_dataset(step_idx, replay_per_class=0)
        eval_ds = make_eval_dataset(classes_for_step(step_idx))

        out_dir = os.path.join(
            MODELS_DIR,
            f"seq_ft_no_replay_step_{step_idx + 1}",
        )

        print(
            f"\n===== seq_ft_no_replay | "
            f"step {step_idx + 1}/{NUM_STEPS} ====="
        )

        train_with_trainer(
            model=seq_ft_model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=out_dir,
            epochs=FT_EPOCHS,
            lr=LR_FT,
            batch_size=BATCH_FT,
            accum_steps=ACCUM_FT,
        )

    evaluate_model(seq_ft_model, "seq_ft_no_replay")

    del seq_ft_model
    cleanup()

else:
    print("Skipping seq_ft_no_replay")

In [ ]:
# ============================================================
# 10) Train independent LoRAs
# ============================================================

def train_independent_loras(method_prefix, replay_per_class=0):
    """
    Train one independent LoRA per step.

    Used for:
    - simple_avg_no_replay
    - simple_avg_replay
    - do_merging_simple
    """
    step_states = []

    for step_idx in range(NUM_STEPS):
        model = fresh_pretrained_model()
        model = add_lora(model)

        model.print_trainable_parameters()

        train_ds = make_train_dataset(
            step_idx=step_idx,
            replay_per_class=replay_per_class,
        )

        eval_ds = make_eval_dataset(classes_for_step(step_idx))

        out_dir = os.path.join(
            MODELS_DIR,
            f"{method_prefix}_step_{step_idx + 1}",
        )

        print(
            f"\n===== {method_prefix} | "
            f"step {step_idx + 1}/{NUM_STEPS} | "
            f"replay_per_class={replay_per_class} ====="
        )

        train_with_trainer(
            model=model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=out_dir,
            epochs=LORA_EPOCHS,
            lr=LR_LORA,
            batch_size=BATCH_LORA,
            accum_steps=ACCUM_LORA,
        )

        state = extract_lora_state(model)
        step_states.append(state)

        del model
        cleanup()

    return step_states

In [ ]:
# ============================================================
# 11) simple_avg_no_replay
# ============================================================

step_states_no_replay = None

if METHODS_TO_RUN["simple_avg_no_replay"] or METHODS_TO_RUN["do_merging_simple"]:
    step_states_no_replay = train_independent_loras(
        method_prefix="simple_avg_no_replay_source",
        replay_per_class=0,
    )

    simple_delta = simple_average_deltas(step_states_no_replay)
    simple_model = apply_deltas_to_base(
        merged_deltas=simple_delta,
        step_states=step_states_no_replay,
    )

    evaluate_model(simple_model, "simple_avg_no_replay")

    del simple_model
    cleanup()

else:
    print("Skipping simple_avg_no_replay")

In [ ]:
# ============================================================
# 12) simple_avg_replay
# ============================================================

step_states_replay = None

if METHODS_TO_RUN["simple_avg_replay"]:
    step_states_replay = train_independent_loras(
        method_prefix="simple_avg_replay_source",
        replay_per_class=REPLAY_PER_CLASS,
    )

    replay_delta = simple_average_deltas(step_states_replay)
    replay_model = apply_deltas_to_base(
        merged_deltas=replay_delta,
        step_states=step_states_replay,
    )

    evaluate_model(replay_model, "simple_avg_replay")

    del replay_model
    cleanup()

else:
    print("Skipping simple_avg_replay")

In [ ]:
# ============================================================
# 13) do_merging_simple
# ============================================================

if METHODS_TO_RUN["do_merging_simple"]:
    assert step_states_no_replay is not None

    do_delta = do_merge_deltas(step_states_no_replay)
    do_model = apply_deltas_to_base(
        merged_deltas=do_delta,
        step_states=step_states_no_replay,
    )

    evaluate_model(do_model, "do_merging_simple")

    del do_model
    cleanup()

else:
    print("Skipping do_merging_simple")

In [ ]:
# ============================================================
# 14) orthogonal_loss
# ============================================================

# Supervisor's idea:
# After each step, absorb the trained LoRA into the model.
# For the next step, train a new LoRA with:
# L = CE + lambda_orth * mean_i tr(M_(t-1)^i (Delta W_t^i)^T)
#
# Here M_(t-1)^i is the previous absorbed model weight for the same q_proj/v_proj layer.
# Delta W_t^i is the current LoRA update B @ A for that layer.
# For numerical stability, the implemented penalty uses a normalized trace/dot product.


def extract_reference_weights_for_orth(peft_model):
    """
    Extract M_(t-1) for every current LoRA target module.
    These are the base q_proj/v_proj weights before training the current LoRA.
    """
    reference_weights = {}

    for name, module in peft_model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        plain_name = normalize_module_name(name)

        if hasattr(module, "base_layer") and hasattr(module.base_layer, "weight"):
            reference_weights[plain_name] = module.base_layer.weight.detach().cpu().float().clone()
        elif hasattr(module, "weight"):
            reference_weights[plain_name] = module.weight.detach().cpu().float().clone()

    return reference_weights


def compute_supervisor_orth_penalty(model, reference_weights, eps=1e-8):
    """
    Mean normalized trace between previous absorbed model weights and current LoRA deltas.

    Supervisor formula per layer:
        tr(M_(t-1) DeltaW_t^T)

    Since both matrices have the same shape, this is equivalent to:
        sum(M_(t-1) * DeltaW_t)

    We normalize by matrix norms to keep the scale stable during debug runs.
    """
    penalties = []
    device = next(model.parameters()).device

    for name, module in model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        plain_name = normalize_module_name(name)

        if plain_name not in reference_weights:
            continue

        A = module.lora_A["default"].weight
        B = module.lora_B["default"].weight

        scaling = (
            module.scaling["default"]
            if isinstance(module.scaling, dict)
            else module.scaling
        )

        delta = (B @ A) * float(scaling)
        old_weight = reference_weights[plain_name].to(
            device=delta.device,
            dtype=delta.dtype,
        )

        trace_value = torch.sum(old_weight * delta)
        normalized_trace = trace_value / (
            torch.linalg.norm(old_weight).clamp_min(eps)
            * torch.linalg.norm(delta).clamp_min(eps)
        )

        penalties.append(normalized_trace.pow(2))

    if not penalties:
        return torch.tensor(0.0, device=device)

    return torch.stack(penalties).mean()


class OrthogonalLossTrainer(Trainer):
    """
    Trainer for supervisor's orthogonal_loss variant.

    It adds an orthogonality-style penalty between:
    - previous absorbed model weights M_(t-1)
    - current LoRA delta DeltaW_t
    """

    def __init__(self, *args, reference_weights=None, lambda_orth=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.reference_weights = reference_weights or {}
        self.lambda_orth = float(lambda_orth)

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        outputs = model(**inputs)
        ce_loss = outputs.loss

        orth_loss = compute_supervisor_orth_penalty(
            model=model,
            reference_weights=self.reference_weights,
        )

        loss = ce_loss + self.lambda_orth * orth_loss

        return (loss, outputs) if return_outputs else loss


if METHODS_TO_RUN["orthogonal_loss"]:
    # Start from CLIP-ViT and sequentially absorb each trained LoRA.
    orth_model = fresh_pretrained_model()

    for step_idx in range(NUM_STEPS):
        print(f"\n===== orthogonal_loss | step {step_idx + 1}/{NUM_STEPS} =====")

        # Add a fresh LoRA on top of the previous absorbed model.
        orth_peft_model = add_lora(orth_model)
        orth_peft_model.print_trainable_parameters()

        reference_weights = extract_reference_weights_for_orth(orth_peft_model)

        train_ds = make_train_dataset(step_idx, replay_per_class=0)
        eval_ds = make_eval_dataset(classes_for_step(step_idx))

        train_with_trainer(
            model=orth_peft_model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=os.path.join(MODELS_DIR, f"orthogonal_loss_step_{step_idx + 1}"),
            epochs=ORTH_EPOCHS,
            lr=LR_ORTH,
            batch_size=BATCH_LORA,
            accum_steps=ACCUM_LORA,
            trainer_cls=OrthogonalLossTrainer,
            reference_weights=reference_weights,
            lambda_orth=LAMBDA_ORTH,
        )

        # Absorb the trained LoRA into the model before the next step.
        orth_model = orth_peft_model.merge_and_unload()

        del orth_peft_model
        cleanup()

    evaluate_model(orth_model, "orthogonal_loss")

    del orth_model
    cleanup()

else:
    print("Skipping orthogonal_loss")

In [ ]:
# ============================================================
# 15) rank_extension
# ============================================================

# Pietro/Leon intended version:
# - only one LoRA exists conceptually
# - its rank grows step by step
# - old rank parts are copied and frozen
# - only the new rank block is trained
#
# Example with RANKEXT_NEW_RANK_PER_STEP = 8:
# Step 1: rank 8
# Step 2: rank 16, old rank-8 block copied and frozen
# Step 3: rank 24, old rank-16 block copied and frozen
# Step 4: rank 32, old rank-24 block copied and frozen
# Step 5: rank 40, old rank-32 block copied and frozen


class GrowingRankLoRALinear(nn.Module):
    """
    Linear layer with a growing-rank LoRA update.

    The frozen old part is stored separately from the new trainable part.
    Only A_new and B_new are trainable in the current step.
    """

    def __init__(
        self,
        base_layer,
        total_rank,
        frozen_rank,
        alpha,
        dropout,
        old_A=None,
        old_B=None,
    ):
        super().__init__()

        assert isinstance(base_layer, nn.Linear)
        assert 0 <= frozen_rank <= total_rank

        self.base_layer = base_layer
        self.total_rank = int(total_rank)
        self.frozen_rank = int(frozen_rank)
        self.new_rank = int(total_rank - frozen_rank)
        self.alpha = float(alpha)
        self.scaling = self.alpha / float(self.total_rank)
        self.dropout = nn.Dropout(dropout)

        in_features = base_layer.in_features
        out_features = base_layer.out_features

        # Freeze the original CLIP projection weights.
        for p in self.base_layer.parameters():
            p.requires_grad = False

        if self.frozen_rank > 0:
            assert old_A is not None and old_B is not None
            assert old_A.shape == (self.frozen_rank, in_features)
            assert old_B.shape == (out_features, self.frozen_rank)

            self.A_frozen = nn.Parameter(old_A.clone().float(), requires_grad=False)
            self.B_frozen = nn.Parameter(old_B.clone().float(), requires_grad=False)
        else:
            self.A_frozen = None
            self.B_frozen = None

        if self.new_rank > 0:
            self.A_new = nn.Parameter(torch.zeros(self.new_rank, in_features))
            self.B_new = nn.Parameter(torch.zeros(out_features, self.new_rank))

            # Common LoRA initialization: A random, B zero.
            nn.init.kaiming_uniform_(self.A_new, a=np.sqrt(5))
            nn.init.zeros_(self.B_new)
        else:
            self.A_new = None
            self.B_new = None

    def full_A_B(self):
        A_parts = []
        B_parts = []

        if self.A_frozen is not None:
            A_parts.append(self.A_frozen)
            B_parts.append(self.B_frozen)

        if self.A_new is not None:
            A_parts.append(self.A_new)
            B_parts.append(self.B_new)

        A = torch.cat(A_parts, dim=0)
        B = torch.cat(B_parts, dim=1)
        return A, B

    def forward(self, x):
        base_out = self.base_layer(x)

        A, B = self.full_A_B()
        lora_hidden = F.linear(self.dropout(x), A)
        lora_out = F.linear(lora_hidden, B)

        return base_out + self.scaling * lora_out


def set_submodule_by_name(model, module_name, new_module):
    parts = module_name.split(".")
    parent = model

    for part in parts[:-1]:
        parent = getattr(parent, part)

    setattr(parent, parts[-1], new_module)


def get_target_linear_module_names(model):
    """
    CLIP-ViT target modules for rank-extension.
    We use q_proj and v_proj, same as PEFT LoRA.
    """
    names = []

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and any(name.endswith(t) for t in TARGET_MODULES):
            names.append(name)

    return names


def build_rank_extension_model(previous_state=None, previous_classifier=None, step_idx=0):
    """
    Build CLIP-ViT with growing-rank LoRA modules.
    Previous A/B blocks are copied and frozen.
    """
    model = fresh_pretrained_model()

    if previous_classifier is not None:
        with torch.no_grad():
            model.classifier.weight.copy_(previous_classifier["weight"])
            model.classifier.bias.copy_(previous_classifier["bias"])

    total_rank = (step_idx + 1) * RANKEXT_NEW_RANK_PER_STEP
    frozen_rank = step_idx * RANKEXT_NEW_RANK_PER_STEP

    target_names = get_target_linear_module_names(model)

    for name in target_names:
        base_layer = get_submodule_by_name(model, name)

        old_A = None
        old_B = None

        if previous_state is not None:
            old_A = previous_state[name]["A"]
            old_B = previous_state[name]["B"]

        new_module = GrowingRankLoRALinear(
            base_layer=base_layer,
            total_rank=total_rank,
            frozen_rank=frozen_rank,
            alpha=LORA_ALPHA,
            dropout=LORA_DROPOUT,
            old_A=old_A,
            old_B=old_B,
        )

        set_submodule_by_name(model, name, new_module)

    # Freeze everything first.
    for p in model.parameters():
        p.requires_grad = False

    # Train only the new LoRA block and classifier.
    for module in model.modules():
        if isinstance(module, GrowingRankLoRALinear):
            if module.A_new is not None:
                module.A_new.requires_grad = True
            if module.B_new is not None:
                module.B_new.requires_grad = True

    for p in model.classifier.parameters():
        p.requires_grad = True

    return model


def extract_rank_extension_state(model):
    """
    Extract full A/B from every growing-rank LoRA module.
    This full A/B becomes the frozen prefix in the next step.
    """
    state = {}

    for name, module in model.named_modules():
        if isinstance(module, GrowingRankLoRALinear):
            A, B = module.full_A_B()
            state[name] = {
                "A": A.detach().cpu().clone(),
                "B": B.detach().cpu().clone(),
            }

    classifier = {
        "weight": model.classifier.weight.detach().cpu().clone(),
        "bias": model.classifier.bias.detach().cpu().clone(),
    }

    return state, classifier


def add_classifier_row_gradient_mask(model, current_classes):
    """
    Freeze classifier rows outside the current step.
    This prevents old class rows from drifting during the next step.
    """
    mask_w = torch.zeros_like(model.classifier.weight)
    mask_b = torch.zeros_like(model.classifier.bias)

    current_classes = list(current_classes)
    mask_w[current_classes, :] = 1.0
    mask_b[current_classes] = 1.0

    mask_w = mask_w.to(model.classifier.weight.device)
    mask_b = mask_b.to(model.classifier.bias.device)

    hook_w = model.classifier.weight.register_hook(lambda grad: grad * mask_w)
    hook_b = model.classifier.bias.register_hook(lambda grad: grad * mask_b)

    return [hook_w, hook_b]


if METHODS_TO_RUN["rank_extension"]:
    previous_rank_state = None
    previous_classifier = None
    rankext_model = None

    for step_idx in range(NUM_STEPS):
        print(f"\n===== rank_extension | step {step_idx + 1}/{NUM_STEPS} =====")
        print("Total rank:", (step_idx + 1) * RANKEXT_NEW_RANK_PER_STEP)
        print("Frozen old rank:", step_idx * RANKEXT_NEW_RANK_PER_STEP)
        print("Trainable new rank:", RANKEXT_NEW_RANK_PER_STEP)

        rankext_model = build_rank_extension_model(
            previous_state=previous_rank_state,
            previous_classifier=previous_classifier,
            step_idx=step_idx,
        )

        trainable = sum(p.numel() for p in rankext_model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in rankext_model.parameters())
        print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.4f}%)")

        hooks = add_classifier_row_gradient_mask(
            rankext_model,
            current_classes=classes_for_step(step_idx),
        )

        train_ds = make_train_dataset(step_idx, replay_per_class=0)
        eval_ds = make_eval_dataset(classes_for_step(step_idx))

        train_with_trainer(
            model=rankext_model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=os.path.join(MODELS_DIR, f"rank_extension_step_{step_idx + 1}"),
            epochs=RANKEXT_EPOCHS,
            lr=LR_RANKEXT,
            batch_size=BATCH_LORA,
            accum_steps=ACCUM_LORA,
        )

        for h in hooks:
            h.remove()

        previous_rank_state, previous_classifier = extract_rank_extension_state(rankext_model)

        if step_idx < NUM_STEPS - 1:
            del rankext_model
            cleanup()

    evaluate_model(rankext_model, "rank_extension")

    del rankext_model
    cleanup()

else:
    print("Skipping rank_extension")

In [ ]:
# ============================================================
# 14) joint_upper_bound
# ============================================================

if METHODS_TO_RUN["joint_upper_bound"]:
    joint_model = fresh_pretrained_model()

    train_joint = make_joint_train_dataset()
    test_joint = make_joint_eval_dataset()

    print("\n===== joint_upper_bound =====")

    train_with_trainer(
        model=joint_model,
        train_ds=train_joint,
        eval_ds=test_joint,
        output_dir=os.path.join(MODELS_DIR, "joint_upper_bound"),
        epochs=JOINT_EPOCHS,
        lr=LR_JOINT,
        batch_size=BATCH_FT,
        accum_steps=ACCUM_FT,
    )

    evaluate_model(joint_model, "joint_upper_bound")

    del joint_model
    cleanup()

else:
    print("Skipping joint_upper_bound")

In [ ]:
# ============================================================
# 15) Final results table
# ============================================================

results_df = pd.DataFrame(all_results)

results_path = os.path.join(TABLES_DIR, "all_results_clip_vit_debug.csv")
results_df.to_csv(results_path, index=False)

print("Saved:", results_path)
results_df

In [ ]:
# ============================================================
# 18) Final pivot table + summary metrics
# ============================================================

method_order = [
    "seq_ft_no_replay",
    "simple_avg_no_replay",
    "simple_avg_replay",
    "do_merging_simple",
    "orthogonal_loss",
    "rank_extension",
    "joint_upper_bound",
]

results_df = pd.DataFrame(all_results)

results_path = os.path.join(TABLES_DIR, "all_results_clip_vit_full_comparison.csv")
results_df.to_csv(results_path, index=False)

print("Saved raw results:", results_path)
display(results_df)

final_table = results_df.pivot_table(
    index="method",
    columns="eval_set",
    values="accuracy",
    aggfunc="mean",
)

for col in ["first_step", "later_steps", "all_seen"]:
    if col not in final_table.columns:
        final_table[col] = np.nan

final_table = final_table.reindex(
    [m for m in method_order if m in final_table.index]
)

final_table = final_table[
    ["first_step", "later_steps", "all_seen"]
]

final_table_percent = final_table * 100

# Extra comparison metrics
summary_table = final_table_percent.copy()

summary_table["old_new_gap"] = (
    summary_table["first_step"] - summary_table["later_steps"]
)

summary_table["min_old_new"] = summary_table[
    ["first_step", "later_steps"]
].min(axis=1)

summary_table["avg_old_new"] = summary_table[
    ["first_step", "later_steps"]
].mean(axis=1)

# Difference from joint upper bound
if "joint_upper_bound" in summary_table.index:
    joint_all_seen = summary_table.loc["joint_upper_bound", "all_seen"]
    summary_table["gap_to_joint_all_seen"] = (
        joint_all_seen - summary_table["all_seen"]
    )
else:
    summary_table["gap_to_joint_all_seen"] = np.nan

# Replay gain
if (
    "simple_avg_no_replay" in summary_table.index
    and "simple_avg_replay" in summary_table.index
):
    replay_gain = (
        summary_table.loc["simple_avg_replay", ["first_step", "later_steps", "all_seen"]]
        - summary_table.loc["simple_avg_no_replay", ["first_step", "later_steps", "all_seen"]]
    )

    print("\nReplay gain: simple_avg_replay - simple_avg_no_replay")
    print(replay_gain.round(2))

final_table_path = os.path.join(TABLES_DIR, "final_accuracy_clip_vit_percent.csv")
summary_table_path = os.path.join(TABLES_DIR, "summary_metrics_clip_vit_percent.csv")

final_table_percent.to_csv(final_table_path)
summary_table.to_csv(summary_table_path)

print("\nSaved final accuracy table:", final_table_path)
print("Saved summary metrics table:", summary_table_path)

print("\nFinal accuracy (%):")
display(final_table_percent.round(2))

print("\nSummary metrics (%):")
display(summary_table.round(2))

In [ ]:
# ============================================================
# 19) More plots for comparison
# ============================================================

plot_df = final_table_percent.reset_index()

# ------------------------------------------------------------
# Plot 1: grouped bar accuracy
# ------------------------------------------------------------
x = np.arange(len(plot_df))
width = 0.25

plt.figure(figsize=(13, 6))

plt.bar(x - width, plot_df["first_step"], width, label="first_step")
plt.bar(x, plot_df["later_steps"], width, label="later_steps")
plt.bar(x + width, plot_df["all_seen"], width, label="all_seen")

plt.xticks(x, plot_df["method"], rotation=30, ha="right")
plt.ylabel("Accuracy (%)")
plt.title("CLIP-ViT + Split CIFAR-100: Final Accuracy Comparison")
plt.legend()
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "01_grouped_accuracy_comparison.png")
plt.savefig(plot_path, dpi=200)
plt.show()
print("Saved:", plot_path)


# ------------------------------------------------------------
# Plot 2: heatmap accuracy
# ------------------------------------------------------------
plt.figure(figsize=(9, 5))

heatmap_data = final_table_percent[
    ["first_step", "later_steps", "all_seen"]
].copy()

plt.imshow(heatmap_data.values, aspect="auto")
plt.colorbar(label="Accuracy (%)")

plt.xticks(
    ticks=np.arange(len(heatmap_data.columns)),
    labels=heatmap_data.columns,
    rotation=0,
)

plt.yticks(
    ticks=np.arange(len(heatmap_data.index)),
    labels=heatmap_data.index,
)

for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        value = heatmap_data.iloc[i, j]
        if not np.isnan(value):
            plt.text(j, i, f"{value:.1f}", ha="center", va="center")

plt.title("Accuracy Heatmap (%)")
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "02_accuracy_heatmap.png")
plt.savefig(plot_path, dpi=200)
plt.show()
print("Saved:", plot_path)


# ------------------------------------------------------------
# Plot 3: all_seen ranking
# ------------------------------------------------------------
ranking_df = final_table_percent.sort_values("all_seen", ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(ranking_df.index, ranking_df["all_seen"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("All-seen accuracy (%)")
plt.title("Method Ranking by All-seen Accuracy")
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "03_all_seen_ranking.png")
plt.savefig(plot_path, dpi=200)
plt.show()
print("Saved:", plot_path)


# ------------------------------------------------------------
# Plot 4: old-new gap
# ------------------------------------------------------------
gap_df = summary_table.copy()

plt.figure(figsize=(10, 5))
plt.bar(gap_df.index, gap_df["old_new_gap"])
plt.axhline(0, linestyle="--", linewidth=1)
plt.xticks(rotation=30, ha="right")
plt.ylabel("first_step - later_steps accuracy (%)")
plt.title("Old-New Accuracy Gap")
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "04_old_new_gap.png")
plt.savefig(plot_path, dpi=200)
plt.show()
print("Saved:", plot_path)


# ------------------------------------------------------------
# Plot 5: gap to joint upper bound
# ------------------------------------------------------------
if "gap_to_joint_all_seen" in summary_table.columns:
    gap_joint_df = summary_table.drop(index="joint_upper_bound", errors="ignore")

    plt.figure(figsize=(10, 5))
    plt.bar(gap_joint_df.index, gap_joint_df["gap_to_joint_all_seen"])
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("Gap to joint all-seen accuracy (%)")
    plt.title("Distance from Joint Upper Bound")
    plt.tight_layout()

    plot_path = os.path.join(PLOTS_DIR, "05_gap_to_joint_upper_bound.png")
    plt.savefig(plot_path, dpi=200)
    plt.show()
    print("Saved:", plot_path)


# ------------------------------------------------------------
# Plot 6: replay gain
# ------------------------------------------------------------
if (
    "simple_avg_no_replay" in final_table_percent.index
    and "simple_avg_replay" in final_table_percent.index
):
    replay_gain_series = (
        final_table_percent.loc["simple_avg_replay", ["first_step", "later_steps", "all_seen"]]
        - final_table_percent.loc["simple_avg_no_replay", ["first_step", "later_steps", "all_seen"]]
    )

    plt.figure(figsize=(7, 5))
    plt.bar(replay_gain_series.index, replay_gain_series.values)
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.ylabel("Accuracy gain (%)")
    plt.title("Replay Gain: simple_avg_replay - simple_avg_no_replay")
    plt.tight_layout()

    plot_path = os.path.join(PLOTS_DIR, "06_replay_gain.png")
    plt.savefig(plot_path, dpi=200)
    plt.show()
    print("Saved:", plot_path)


# ------------------------------------------------------------
# Plot 7: loss comparison
# ------------------------------------------------------------
loss_table = results_df.pivot_table(
    index="method",
    columns="eval_set",
    values="loss",
    aggfunc="mean",
)

loss_table = loss_table.reindex(
    [m for m in method_order if m in loss_table.index]
)

for col in ["first_step", "later_steps", "all_seen"]:
    if col not in loss_table.columns:
        loss_table[col] = np.nan

loss_table = loss_table[
    ["first_step", "later_steps", "all_seen"]
]

loss_table_path = os.path.join(TABLES_DIR, "final_loss_table.csv")
loss_table.to_csv(loss_table_path)

loss_plot_df = loss_table.reset_index()

x = np.arange(len(loss_plot_df))
width = 0.25

plt.figure(figsize=(13, 6))

plt.bar(x - width, loss_plot_df["first_step"], width, label="first_step")
plt.bar(x, loss_plot_df["later_steps"], width, label="later_steps")
plt.bar(x + width, loss_plot_df["all_seen"], width, label="all_seen")

plt.xticks(x, loss_plot_df["method"], rotation=30, ha="right")
plt.ylabel("Loss")
plt.title("Evaluation Loss Comparison")
plt.legend()
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "07_loss_comparison.png")
plt.savefig(plot_path, dpi=200)
plt.show()

print("Saved loss table:", loss_table_path)
print("Saved:", plot_path)

In [ ]:
# ============================================================
# 20) Explanation for supervisor
# ============================================================

print(f"""
Supervisor-aligned full comparison setup:

Backbone:
- CLIP-ViT vision encoder: {MODEL_CHECKPOINT}
- Text encoder is not used.
- A new CIFAR-100 classifier is trained on top of the CLIP vision representation.
- LoRA target modules are q_proj and v_proj.

Continual learning setup:
- Split CIFAR-100
- 5 steps
- 20 classes per step
- Evaluation sets:
  1. first_step   = classes 0-19
  2. later_steps  = classes 20-99
  3. all_seen     = classes 0-99

Compared methods:
1. seq_ft_no_replay
   Sequential full fine-tuning without replay.
   This is the catastrophic-forgetting baseline.

2. simple_avg_no_replay
   One independent LoRA trained per step without replay.
   The LoRA deltas are merged using simple average.
   This is the main no-replay merging method.

3. simple_avg_replay
   Same as simple_avg_no_replay, but each step uses a small replay buffer.
   Replay is used only as a diagnostic to measure the effect of old-data exposure.

4. do_merging_simple
   Simplified DO-Merging-inspired variant.
   It decouples magnitude and direction before merging.
   This is not a full reproduction of the DO-Merging paper yet.

5. orthogonal_loss
   Supervisor-suggested orthogonality variant.
   After each step, the trained LoRA is absorbed into the model.
   The next LoRA is trained with an orthogonality-style loss against the previous absorbed model weights.

6. rank_extension
   Pietro/Leon rank-extension variant.
   One LoRA grows in rank step by step.
   Old rank parts are copied and frozen; only the new rank block is trained.

7. joint_upper_bound
   Joint training on all classes.
   This is not a continual-learning method; it is the upper bound reference.

Epochs:
- FT_EPOCHS = {FT_EPOCHS}
- LORA_EPOCHS = {LORA_EPOCHS}
- JOINT_EPOCHS = {JOINT_EPOCHS}
- ORTH_EPOCHS = {ORTH_EPOCHS}
- RANKEXT_EPOCHS = {RANKEXT_EPOCHS}

Plots saved:
1. grouped accuracy comparison
2. accuracy heatmap
3. all-seen ranking
4. old-new gap
5. gap to joint upper bound
6. replay gain
7. loss comparison

Main interpretation to check:
- Does simple_avg_no_replay beat seq_ft_no_replay?
- Does simple_avg_replay improve over simple_avg_no_replay?
- How far are methods from joint_upper_bound?
- Do orthogonal_loss or rank_extension improve old-class retention or all-seen accuracy?
""")